In [1]:
!pip install -q chromadb langchain-text-splitters groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
# ===========================================================================
# CELL 2: API Credentials Setup
# ===========================================================================
import os
from getpass import getpass
GROQ_API_KEY = getpass('Enter your Groq API key (from console.groq.com): ')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print("API Key set successfully.")

Enter your Groq API key (from console.groq.com): ··········
API Key set successfully.


In [3]:
# ===========================================================================
# CELL 3: Unified Engine & Evaluation Setup
# ===========================================================================

import os
import re
import warnings
import numpy as np
from groq import Groq
import chromadb
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# PATHS (detects environment to support both Colab and local execution)
# ---------------------------------------------------------------------------
if IN_COLAB:
    CHROMA_DB_DIR = "/content/chroma_db_grap_dhaka"
    OUTPUT_DIR    = "/content/Figures"
    NAQMP_MD_PATH = "/content/Bangladesh National Air Quality Management Plan 2024-2030.md"
    WHO_MD_PATH   = "/content/WHO global_Air_Quality_Guildlines_eng.md"
    APCR_MD_PATH  = "/content/Air Pollution Control Rules 2022.md"
else:
    # Local Windows Workspace Paths
    # Current script resides in: <workspace>/03_Code/Final RUN/
    BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    CHROMA_DB_DIR = os.path.join(BASE_DIR, "chroma_db_grap_dhaka")
    OUTPUT_DIR    = os.path.join(BASE_DIR, "Final RUN", "EAAB")
    NAQMP_MD_PATH = os.path.join(BASE_DIR, "policy_corpus", "Markdown_version_of_the_Pdf", "Bangladesh National Air Quality Management Plan 2024-2030.md")
    WHO_MD_PATH   = os.path.join(BASE_DIR, "policy_corpus", "Markdown_version_of_the_Pdf", "WHO global_Air_Quality_Guildlines_eng.md")
    APCR_MD_PATH  = os.path.join(BASE_DIR, "policy_corpus", "Markdown_version_of_the_Pdf", "Air Pollution Control Rules 2022.md")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===========================================================================
# EVALUATION CONFIGURATION — Calibrated for all-MiniLM-L6-v2 Embeddings
# ===========================================================================
EVAL_CONFIG = {
    "embedding_model":          "all-MiniLM-L6-v2 (ChromaDB built-in, 384-dim)",
    "faithfulness_threshold":   0.42,
    "relevance_threshold":      0.50,
    "precision_threshold":      0.80,
    "hallucination_threshold":  0.05,
    "baseline_context_recall":  0.797,
}

# ===========================================================================
# ENGINE CLASS
# ===========================================================================

class GRAPDhakaEngine:
    """
    The Policy-Auditing RAG Engine for GRAP-Dhaka.
    Bridges XGBoost PM2.5 forecasts + SHAP drivers with the NAQMP and WHO
    statutory corpora to produce cited Environmental Action Advisory Briefs.
    """

    def __init__(self, api_key: str, rebuild: bool = False):
        self.api_key     = api_key
        self.groq_client = Groq(api_key=api_key)
        self.chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)

        if rebuild:
            print("[DB] Rebuilding collections from scratch...")
            for name in ["naqmp_policy", "who_guidelines"]:
                try:
                    self.chroma_client.delete_collection(name)
                    print(f"[DB]   Deleted: {name}")
                except Exception:
                    pass

        self.naqmp_col = self.chroma_client.get_or_create_collection(
            name="naqmp_policy",
            metadata={"hnsw:space": "cosine"}
        )
        self.who_col = self.chroma_client.get_or_create_collection(
            name="who_guidelines",
            metadata={"hnsw:space": "cosine"}
        )

    def _extract_page_from_chunk(self, text: str) -> int:
        # Pattern 1: Gazette-style page headers like '###### 12747' (used in APCR 2022)
        headings = re.findall(r'######\s*(\d+)', text)
        if headings:
            return int(headings[-1])
        # Pattern 2: Explicit 'page N' references within text
        standards = re.findall(r'(?:page|pg\.?)\s*\|?\s*\[?(\d+)\]?', text, re.IGNORECASE)
        if standards:
            return int(standards[-1])
        # Pattern 3: Standalone integers on their own line (used in NAQMP 2024-2030 as page footers)
        standalone = re.findall(r'(?m)^\s*(\d+)\s*$', text)
        if standalone:
            # Filter out spurious single-digit matches that are likely list numbers, not pages
            candidates = [int(n) for n in standalone if int(n) >= 5]
            if candidates:
                return candidates[-1]
        return 0

    def _read_and_index_md(self, md_path: str, collection, doc_label: str):
        if not os.path.exists(md_path):
            print(f"[ERROR] Markdown file not found: {md_path}")
            return

        print(f"\n[DB-INDEX] Reading: {os.path.basename(md_path)}")
        with open(md_path, "r", encoding="utf-8") as f:
            md_text = f.read()

        splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
            strip_headers=False
        )
        header_splits = splitter.split_text(md_text)

        chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
        splits = chunk_splitter.split_documents(header_splits)

        if not splits:
            print(f"[DB-INDEX] WARNING: No splits found in {doc_label}. Check Markdown quality.")
            return

        print(f"[DB-INDEX] {len(splits)} semantic chunks extracted. Building local embeddings...")

        documents, metadatas, ids = [], [], []
        current_page = 1

        for idx, split in enumerate(splits):
            content = split.page_content.strip()
            if len(content) < 40:
                continue

            detected_page = self._extract_page_from_chunk(content)
            if detected_page > 0:
                current_page = detected_page

            heading = (split.metadata.get("h2") or
                       split.metadata.get("h1") or
                       split.metadata.get("h3") or
                       "General")

            documents.append(content)
            metadatas.append({
                "source":      doc_label,
                "heading":     str(heading),
                "page_number": current_page,
                "chunk_index": idx,
            })
            ids.append(f"{doc_label}_chunk_{idx}")

        collection.add(documents=documents, metadatas=metadatas, ids=ids)
        print(f"[DB-INDEX] ✅ '{collection.name}' now has {collection.count()} chunks.")

    def build_vector_db(self):
        if self.naqmp_col.count() == 0:
            print("\n[DB] === Building Bangladesh Legal Corpus (Collection 2) ===")
            self._read_and_index_md(NAQMP_MD_PATH, self.naqmp_col, "NAQMP-2024-2030")
            if os.path.exists(APCR_MD_PATH):
                print("\n[DB] ✅ APCR 2022 found. Integrating Tier 2 (Codified Law)...")
                self._read_and_index_md(APCR_MD_PATH, self.naqmp_col, "APCR-2022-SRO")
                print("[DB]    Full legal corpus: Tier 2 (APCR) + Tier 3 (NAQMP).")
            else:
                print("\n[DB] ⚠️  APCR 2022 .md not found. Citations will use NAQMP only.")
        else:
            print(f"[DB] 'naqmp_policy' already has {self.naqmp_col.count()} chunks. Skipping.")

        if self.who_col.count() == 0:
            print("\n[DB] === Building WHO Scientific Baseline (Collection 1) ===")
            self._read_and_index_md(WHO_MD_PATH, self.who_col, "WHO-AQG-2021")
        else:
            print(f"[DB] 'who_guidelines' already has {self.who_col.count()} chunks. Skipping.")

    def classify_grap_stage(self, pm25_forecasts: list) -> tuple:
        rolling_avg = float(np.mean(pm25_forecasts))
        if rolling_avg <= 15.0:
            return ("GREEN",  "Routine",   "Low risk. Complies with WHO 24-hour guideline.",             rolling_avg)
        elif rolling_avg <= 65.0:
            return ("AMBER",  "Alert",     "Moderate risk. Exceeds WHO limit; within Bangladesh NAAQS.", rolling_avg)
        elif rolling_avg <= 150.0:
            return ("RED",    "Emergency", "High risk. Exceeds Bangladesh NAAQS daily standard.",        rolling_avg)
        else:
            return ("PURPLE", "Crisis",    "Severe risk. Extreme South Asian winter pollution episode.", rolling_avg)

    def calculate_crf_risk(self, pm25_val: float) -> float:
        if pm25_val <= 5.0:
            return 0.0
        return (1.0 - np.exp(-0.00575 * (pm25_val - 5.0))) * 100.0

    def run_policy_gap_audit(self, shap_drivers: list, threshold: float = 0.55) -> dict:
        FEATURE_MAP = {
            "pm2_5_mean":    "fine particulate matter PM2.5 daily average ambient air quality standard limits",
            "blh_x_winter":  "planetary boundary layer height winter thermal inversion meteorological emission controls",
            "wind_v_mean":   "meridional wind velocity regional transboundary air pollutant transport air-shed management",
            "aod_extinction": "satellite aerosol optical depth remote sensing monitoring and assimilation air quality",
            "precip_sum":    "precipitation wet deposition scavenging air pollution washout meteorological controls",
        }

        results = {}
        for driver in shap_drivers:
            semantic_query = FEATURE_MAP.get(driver, driver)
            query = (f"emergency mitigation protocols, emission controls, and "
                     f"administrative enforcement guidelines for: {semantic_query}")

            res = self.naqmp_col.query(
                query_texts=[query], n_results=1,
                include=["distances", "documents", "metadatas"]
            )

            distance   = res["distances"][0][0] if res["distances"][0] else 1.0
            similarity = max(0.0, 1.0 - distance)

            top_chunk = res["documents"][0][0] if res["documents"][0] else ""
            top_meta  = res["metadatas"][0][0]  if res["metadatas"][0]  else {}
            source    = top_meta.get("source", "NAQMP")
            citation  = (f"[{source}, Section: {top_meta.get('heading','?')}, "
                         f"p.{top_meta.get('page_number','?')}]")

            if similarity < threshold:
                results[driver] = {
                    "status":     "UNCOVERED GAP",
                    "similarity": round(similarity, 4),
                    "citation":   "N/A",
                    "excerpt":    f"(No matching directive found in {source} corpus.)"
                }
            else:
                results[driver] = {
                    "status":     "COVERED",
                    "similarity": round(similarity, 4),
                    "citation":   citation,
                    "excerpt":    top_chunk[:300]
                }
        return results

    def generate_advisory_brief(self, pm25_forecasts: list, shap_drivers: list) -> tuple:
        t24 = pm25_forecasts[0]
        stage, trigger, description, rolling_avg = self.classify_grap_stage(pm25_forecasts)
        af = self.calculate_crf_risk(rolling_avg)

        # Query NAQMP/APCR for regulatory mitigation directives
        naqmp_res = self.naqmp_col.query(
            query_texts=[f"emergency measures and administrative controls for {trigger} stage air pollution"],
            n_results=3,
            include=["documents", "metadatas"]
        )
        naqmp_context = ""
        for doc, meta in zip(naqmp_res["documents"][0], naqmp_res["metadatas"][0]):
            source   = meta.get("source", "NAQMP")
            heading  = meta.get("heading", "Section Unknown")
            page_num = meta.get("page_number", "?")
            naqmp_context += f"[{source} | {heading} | p.{page_num}]\n{doc}\n\n"

        # Query WHO AQG for health threshold definitions
        who_res = self.who_col.query(
            query_texts=["short-term PM2.5 exposure health risks guideline limit 24-hour"],
            n_results=2,
            include=["documents", "metadatas"]
        )
        who_context = ""
        for doc, meta in zip(who_res["documents"][0], who_res["metadatas"][0]):
            heading  = meta.get("heading", "Section Unknown")
            page_num = meta.get("page_number", "?")
            who_context += f"[WHO AQG 2021 | {heading} | p.{page_num}]\n{doc}\n\n"

        # SHAP-to-RAG policy gap audit
        gap_audit = self.run_policy_gap_audit(shap_drivers)
        gap_lines = []
        for driver, info in gap_audit.items():
            if info["status"] == "UNCOVERED GAP":
                gap_lines.append(
                    f"- **{driver}**: ⚠️ GOVERNANCE GAP (similarity={info['similarity']:.2f}). "
                    f"The NAQMP/APCR contains no enforcement directive for this atmospheric driver."
                )
            else:
                gap_lines.append(
                    f"- **{driver}**: ✅ COVERED (similarity={info['similarity']:.2f}). "
                    f"Nearest citation: {info['citation']}"
                )

        # XGBoost model bias warning (empirically calibrated from Week 5 run)
        bias_warning = ""
        if t24 > 130:
            bias_warning = (f"\n> ⚠️ **Model Bias Warning (Extreme Event):** "
                            f"The XGBoost model systematically underpredicts at this range "
                            f"(mean bias = -39.27 µg/m³). Real-world exposure may approach "
                            f"**{t24 + 39.27:.1f} µg/m³**. All directives should be treated as a conservative lower bound.\n")
        elif t24 > 100:
            bias_warning = (f"\n> ⚠️ **Model Bias Warning (High Event):** "
                            f"Mean underprediction bias = -22.83 µg/m³. "
                            f"Adjusted upper-bound estimate: **{t24 + 22.83:.1f} µg/m³**.\n")

        # Factual Grounding Constants to prevent LLM hallucination
        grounding_constants = """
CRITICAL FACTUAL REGULATORY GROUNDING:
- Bangladesh PM2.5 Ambient Air Quality Standard (Schedule 1 of Air Pollution Control Rules 2022):
  * 24-hour limit: 65 µg/m³
  * Annual limit: 35 µg/m³
- WHO PM2.5 Air Quality Guidelines (WHO AQG 2021):
  * 24-hour limit: 15 µg/m³
  * Annual limit: 5 µg/m³
- Emergency Powers & National Committee on Air Pollution Control (NCAPC):
  * The NCAPC is established and empowered under Rule 15 of the Air Pollution Control Rules 2022 (S.R.O. No. 255-Law/2022, page 12746 and 12747).
  * Directives to close schools, limit outdoor movement, and restrict vehicles or industrial operations during extreme episodes are issued under Rule 15(2)(e) and 15(2)(f) of APCR 2022 SRO, page 12747.
  * Do NOT cite page 28 of the NAQMP 2024-2030 for these NCAPC emergency actions. Page 28 of NAQMP contains Table 3.1 (standards) and does not outline emergency actions.
"""

        prompt = f"""
ROLE: You are a scientific environmental policy advisor producing an official
      Environmental Action Advisory Brief (EAAB) for Dhaka City planners.

{grounding_constants}

STRICT RULES:
- Every operational directive MUST cite a specific section from the retrieved NAQMP/APCR corpus below.
- Every health threshold MUST cite the retrieved WHO AQG text below with page number.
- Do NOT invent legal authority. If no local directive exists, write:
  "Recommended adaptive protocol (NAQMP regulatory vacuum; adapted from regional GRAP precedents)."
- Do NOT estimate absolute death counts. Use only Attributable Fraction (AF%) metrics.
- Use Markdown tables for Sections 1, 3, and 4.
- Do NOT state that the Bangladesh daily NAAQS standard is 60 µg/m³. It is 65 µg/m³.
- Verify that the National Committee on Air Pollution Control (NCAPC) emergency actions cite the Air Pollution Control Rules 2022 (APCR-2022-SRO, Rule 15, page 12747). Do not cite NAQMP 2024-2030 page 28 for these emergency powers.

FORECAST INPUT:
- T+24h Predicted PM2.5: {t24:.2f} µg/m³
- 3-Day Rolling Average: {rolling_avg:.2f} µg/m³
- GRAP-Dhaka Stage: {stage} ({trigger}) — {description}
- CRF Attributable Fraction: {af:.1f}% of acute cardiorespiratory risk
{bias_warning}
RETRIEVED BANGLADESH REGULATORY CORPUS (APCR 2022 & NAQMP 2024-2030):
\"\"\"
{naqmp_context}
\"\"\"

RETRIEVED INTERNATIONAL HEALTH STANDARDS (WHO AQG 2021):
\"\"\"
{who_context}
\"\"\"

COMPUTATIONAL POLICY GAP AUDIT RESULTS:
{chr(10).join(gap_lines)}

OUTPUT: Write a structured Markdown report with exactly these four sections:

### 1. Forecast Diagnostics & Precautionary Calibration
Use a table with columns: Item | Value | Note.
Include modelled PM2.5, 3-day average, GRAP-Dhaka stage, and model bias warning.

### 2. Public Health Exposure Analysis
State the AF% with a plain-English translation (e.g., "X in every 100 people...").
Cite the WHO AQG threshold exceeded, with section and page reference from the retrieved text.

### 3. Graded Operational Directives
Use a table: # | Action | Legal Basis (citation).
List 3–5 specific, actionable steps for tomorrow.
Every action must cite [Source, Section, Page] from the retrieved corpus.
Label actions without local authority as "(Adaptive Protocol)".

### 4. Policy Gap Report
Use a table with columns: Driver | Governance Status | Why It Matters | International Reference.
Include ALL four drivers — both UNCOVERED GAP rows (⚠️) and COVERED rows (✅).
For UNCOVERED GAP rows: explain the legal silence and cite WHO AQG or India GRAP precedent.
For COVERED rows: state which legal instrument covers it and why that coverage is sufficient or partial.

YOUR RESPONSE:
"""
        chat = self.groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="openai/gpt-oss-120b",
            temperature=0.3
        )
        eval_context = {
            "naqmp_docs":  naqmp_res["documents"][0],
            "naqmp_metas": naqmp_res["metadatas"][0],
            "who_docs":    who_res["documents"][0],
            "who_metas":   who_res["metadatas"][0],
        }
        return chat.choices[0].message.content, eval_context


def run_ragas_evaluation(engine, answer: str, eval_context: dict, diagnostics_json_path: str) -> dict:
    import re as _re
    import numpy as np
    import json

    scores      = {}
    diagnostics = {}

    def _clean_spaces(text: str) -> str:
        text = _re.sub(r'<[^>]+>', '', text)
        text = (text
                .replace('\u2011', '-')
                .replace('\u2013', '-')
                .replace('\u2014', '-'))
        return _re.sub(r'\s+', ' ', text).strip()

    def _norm(text: str) -> str:
        text = text.replace('pm₂.₅', 'pm2.5').replace('pm₂.5', 'pm2.5')
        cleaned = _clean_spaces(text)
        return cleaned.replace('-', ' ').lower()

    all_docs  = eval_context.get("naqmp_docs",  []) + eval_context.get("who_docs",  [])
    all_metas = eval_context.get("naqmp_metas", []) + eval_context.get("who_metas", [])

    def _extract_scoreable_units(text: str, min_len: int = 25) -> list:
        units = []
        for s in _re.split(r'(?<=[.!?])\s+', text):
            s = s.strip()
            if len(s) >= min_len and not s.startswith('|'):
                units.append(s)

        for line in text.split('\n'):
            line = line.strip()
            if line.startswith('|') and '---' not in line:
                cells = [c.strip() for c in line.split('|')]
                for cell in cells:
                    cell = _re.sub(r'\*+', '', cell).strip()
                    if len(cell) >= min_len:
                        units.append(cell)
        return list(dict.fromkeys(units))

    FAITH_THRESHOLD = EVAL_CONFIG["faithfulness_threshold"]
    units = _extract_scoreable_units(answer)
    diagnostics["total_scoreable_units"] = len(units)

    def _lexical_faithfulness(units_list, all_docs_list):
        corpus_text = " ".join(all_docs_list).lower()
        corpus_words = corpus_text.split()
        corpus_4grams = set(
            " ".join(corpus_words[i:i+4]) for i in range(len(corpus_words) - 3)
        )
        matched = 0
        for u in units_list:
            u_words = u.lower().split()
            u_4grams = [" ".join(u_words[i:i+4]) for i in range(len(u_words) - 3)]
            if any(g in corpus_4grams for g in u_4grams):
                matched += 1
        return round(matched / len(units_list), 4) if units_list else 0.0

    diagnostics["faithfulness_lexical_baseline"] = (
        _lexical_faithfulness(units, all_docs) if units else 0.0
    )

    if units:
        supported   = 0
        unit_scores = []
        for unit in units:
            best_sim = 0.0
            for col in [engine.naqmp_col, engine.who_col]:
                try:
                    res = col.query(
                        query_texts=[_clean_spaces(unit[:350])],
                        n_results=1,
                        include=["distances"]
                    )
                    d   = res["distances"][0][0] if res["distances"][0] else 1.0
                    sim = max(0.0, 1.0 - d)
                    best_sim = max(best_sim, sim)
                except Exception:
                    pass
            unit_scores.append(best_sim)
            if best_sim >= FAITH_THRESHOLD:
                supported += 1

        scores["faithfulness"] = round(supported / len(units), 4)
        diagnostics["faithfulness_unit_scores"] = {
            "mean":  round(float(np.mean(unit_scores)), 4),
            "min":   round(float(np.min(unit_scores)),  4),
            "max":   round(float(np.max(unit_scores)),  4),
            "threshold_used": FAITH_THRESHOLD,
            "units_above_threshold": supported,
            "total_units": len(units),
        }
    else:
        scores["faithfulness"] = 0.0

    prose_units = [s for s in _re.split(r'(?<=[.!?])\s+', answer)
                   if len(s.strip()) > 40 and not s.strip().startswith('|')][:6]

    table_units = []
    for line in answer.split('\n'):
        line = line.strip()
        if line.startswith('|') and '---' not in line:
            cells = [_re.sub(r'\*+', '', c).strip()
                     for c in line.split('|') if len(c.strip()) > 40]
            table_units.extend(cells)
    table_units = table_units[:6]

    relevance_units = list(dict.fromkeys(prose_units + table_units))

    if relevance_units:
        sims = []
        for unit in relevance_units:
            best = 0.0
            for col in [engine.naqmp_col, engine.who_col]:
                try:
                    res = col.query(
                        query_texts=[_clean_spaces(unit[:350])],
                        n_results=1,
                        include=["distances"]
                    )
                    d   = res["distances"][0][0] if res["distances"][0] else 1.0
                    sim = max(0.0, 1.0 - d)
                    best = max(best, sim)
                except Exception:
                    pass
            sims.append(best)

        scores["answer_relevance"] = round(float(np.mean(sims)), 4)
        diagnostics["answer_relevance_detail"] = {
            "units_sampled": len(relevance_units),
            "per_unit_sims": [round(s, 3) for s in sims],
            "threshold_used": EVAL_CONFIG["relevance_threshold"],
        }
    else:
        scores["answer_relevance"] = 0.0

    answer_norm = _norm(answer)

    def _chunk_is_cited(meta: dict, answer_text: str) -> tuple:
        src  = _norm(meta.get("source",  ""))
        hdg  = _norm(meta.get("heading", ""))
        page = str(meta.get("page_number", ""))

        if src and len(src) >= 6 and src in answer_text:
            return True, "source_label"
        if page and src and page in answer_text and src in answer_text:
            return True, "page_source_co"

        hdg_words = hdg.split()
        if len(hdg_words) >= 4:
            for i in range(len(hdg_words) - 3):
                window = " ".join(hdg_words[i:i+4])
                if len(window) > 10 and window in answer_text:
                    return True, "heading_window"
        return False, "none"

    cited       = 0
    cite_detail = []
    for meta in all_metas:
        is_cited, method = _chunk_is_cited(meta, answer_norm)
        if is_cited:
            cited += 1
        cite_detail.append({
            "source":  meta.get("source", "?"),
            "heading": meta.get("heading", "?")[:60],
            "page":    meta.get("page_number", "?"),
            "cited":   is_cited,
            "method":  method,
        })

    total = len(all_metas)
    scores["context_precision"] = round(cited / total, 4) if total > 0 else 0.0
    diagnostics["context_precision_detail"] = cite_detail

    valid_sources = {_norm(m.get("source", "")) for m in all_metas}
    valid_sources.update({
        "naqmp", "apcr", "who", "who aqg", "who-aqg-2021",
        "naqmp-2024-2030", "apcr-2022-sro", "who aqg 2021",
        "who global", "bangladesh gazette", "air pollution control"
    })

    bracket_citations = _re.findall(r'\[([^\]]{5,100})\]', answer_norm)
    if bracket_citations:
        hallucinated = sum(
            1 for cite in bracket_citations
            if not any(src in cite for src in valid_sources if src)
        )
        scores["hallucination_rate"] = round(hallucinated / len(bracket_citations), 4)
        diagnostics["hallucination_detail"] = {
            "total_citations_found": len(bracket_citations),
            "hallucinated":          hallucinated,
        }
    else:
        scores["hallucination_rate"] = 0.0

    diagnostics["eval_config_snapshot"] = EVAL_CONFIG

    try:
        with open(diagnostics_json_path, "w", encoding="utf-8") as _jf:
            json.dump(diagnostics, _jf, indent=2, default=str)
        print(f"\n[RAGAS-PROXY] Diagnostics saved → {diagnostics_json_path}")
    except Exception as _je:
        print(f"\n[RAGAS-PROXY] Warning: Could not save diagnostics JSON: {_je}")

    print("\n[RAGAS-PROXY] Evaluation Diagnostics:")
    print(f"  Embedding model           : {EVAL_CONFIG['embedding_model']}")
    print(f"  Scoreable units extracted : {diagnostics.get('total_scoreable_units', 'N/A')}")
    faith_d = diagnostics.get("faithfulness_unit_scores", {})
    if faith_d:
        lex_base = diagnostics.get("faithfulness_lexical_baseline", "N/A")
        print(f"  Faithfulness threshold    : {faith_d['threshold_used']}  (MiniLM-calibrated)")
        print(f"  Units above threshold     : {faith_d['units_above_threshold']} / {faith_d['total_units']}")
        print(f"  Mean unit similarity      : {faith_d['mean']}  (semantic cosine, raw)")
        print(f"  Faithfulness — Lexical    : {lex_base}  [4-gram baseline for comparison]")
    if "answer_relevance_detail" in diagnostics:
        rel_d = diagnostics["answer_relevance_detail"]
        print(f"  Relevance units sampled   : {rel_d['units_sampled']}")
        print(f"  Relevance threshold       : {EVAL_CONFIG['relevance_threshold']}  (MiniLM-calibrated)")
    if "context_precision_detail" in diagnostics:
        print("  Context Precision (per chunk):")
        for row in diagnostics["context_precision_detail"]:
            status = "✅" if row["cited"] else "❌"
            print(f"    {status} [{row['source']} p.{row['page']}] '{row['heading'][:50]}' ({row['method']})")

    return scores


def _normalize_pm_notation(text: str) -> str:
    import re as _re
    return _re.sub(r'PM[₂2]\.?[₅5]', 'PM₂.₅', text)


def print_ragas_results(scores: dict):
    THRESHOLDS = {
        "faithfulness":       EVAL_CONFIG["faithfulness_threshold"],
        "answer_relevance":   EVAL_CONFIG["relevance_threshold"],
        "context_precision":  EVAL_CONFIG["precision_threshold"],
        "hallucination_rate": EVAL_CONFIG["hallucination_threshold"],
    }
    LABELS = {
        "faithfulness":      "Faithfulness (Semantic)",
        "answer_relevance":  "Answer Relevance",
        "context_precision": "Context Precision",
        "hallucination_rate":"Hallucination Rate",
    }

    print("\n[PHASE 3] RAGAS Evaluation — Live Computed Scores")
    print(f"          Embedding: {EVAL_CONFIG['embedding_model']}")
    print("          Thresholds calibrated to MiniLM cosine space (see EVAL_CONFIG).")
    print("-" * 86)
    print(f"{'RAGAS Metric':<27} {'Threshold':<14} {'Computed Score':<16} {'Status'}")
    print("-" * 86)

    all_pass = True
    for key, label in LABELS.items():
        threshold = THRESHOLDS[key]
        computed  = scores.get(key, 0.0)
        if key == "hallucination_rate":
            passed     = computed <= threshold
            thresh_str = f"<= {threshold*100:.0f}%"
            score_str  = f"{computed*100:.1f}%"
        else:
            passed     = computed >= threshold
            thresh_str = f">= {threshold:.2f}"
            score_str  = f"{computed:.3f}"
        status = "✅ PASS" if passed else "⚠️  BELOW TARGET"
        if not passed:
            all_pass = False
        print(f"{label:<27} {thresh_str:<14} {score_str:<16} {status}")

    print("-" * 86)
    print(f"\nBaseline: Gao et al. (2024) Advanced RAG — Context Recall = {EVAL_CONFIG['baseline_context_recall']}")
    print("Note: Thresholds are MiniLM-calibrated (raw cosine). No scaling multipliers applied.")

    if all_pass:
        print("\n✅ All RAGAS-proxy metrics satisfy the calibrated evaluation thresholds.")
    else:
        print("\n⚠️  One or more RAGAS-proxy metrics fall below the calibrated threshold.")
    print()


# ===========================================================================
# SCENARIOS EXECUTION
# ===========================================================================

def run_scenarios(engine):
    scenarios = [
        {
            "name": "Winter Inversion Crisis",
            "filename": "EAAB_Winter_Inversion_Scenario.md",
            "diagnostics": "EAAB_eval_diagnostics.json",
            "forecasts": [145.0, 155.0, 165.0],
            "drivers": ["pm2_5_mean", "blh_x_winter", "wind_v_mean", "aod_extinction"],
            "title_header": "# GRAP-Dhaka EAAB — Winter Inversion Scenario\n\n**Forecast:** T+24h = 145.0 µg/m³  |  Stage: PURPLE (Crisis)\n\n**Generated by:** GRAP-Dhaka Policy-Auditing RAG Agent (Week 7)\n\n---\n\n"
        },
        {
            "name": "Monsoon Washout",
            "filename": "EAAB_Monsoon_Scenario.md",
            "diagnostics": "EAAB_Monsoon_eval_diagnostics.json",
            "forecasts": [28.0, 25.0, 22.0],
            "drivers": ["pm2_5_mean", "precip_sum", "wind_v_mean", "aod_extinction"],
            "title_header": "# GRAP-Dhaka EAAB — Monsoon Washout Scenario\n\n**Forecast:** T+24h = 28.0 µg/m³  |  Stage: AMBER (Alert)\n\n**Physics:** Precipitation wet-scavenging + southerly monsoon winds\n\n**Generated by:** GRAP-Dhaka Policy-Auditing RAG Agent (Week 7)\n\n---\n\n"
        },
        {
            "name": "Extreme Pollution Episode",
            "filename": "EAAB_Extreme_Scenario.md",
            "diagnostics": "EAAB_Extreme_eval_diagnostics.json",
            "forecasts": [195.0, 205.0, 215.0],
            "drivers": ["pm2_5_mean", "blh_x_winter", "wind_v_mean", "aod_extinction"],
            "title_header": "# GRAP-Dhaka EAAB — Extreme Pollution Episode\n\n**Forecast:** T+24h = 195.0 µg/m³  |  Stage: PURPLE (Crisis — ceiling)\n\n**Bias-adjusted exposure:** ~234 µg/m³ (XGBoost underprediction bias = -39.27)\n\n**Generated by:** GRAP-Dhaka Policy-Auditing RAG Agent (Week 7)\n\n---\n\n"
        }
    ]

    for sc in scenarios:
        print("\n" + "=" * 65)
        print(f" Running Scenario: {sc['name']}")
        print("=" * 65)

        advisory_brief, eval_ctx = engine.generate_advisory_brief(sc["forecasts"], sc["drivers"])
        advisory_brief = _normalize_pm_notation(advisory_brief)

        print("\n" + "=" * 65)
        print("  ENVIRONMENTAL ACTION ADVISORY BRIEF (EAAB)")
        print("=" * 65)
        print(advisory_brief)
        print("=" * 65)

        out_path = os.path.join(OUTPUT_DIR, sc["filename"])
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(sc["title_header"])
            f.write(advisory_brief)
        print(f"\n[+] Brief saved to: {out_path}")

        print("\n[PHASE 3] Computing RAGAS metrics from live run...")
        diag_path = os.path.join(OUTPUT_DIR, sc["diagnostics"])
        ragas_scores = run_ragas_evaluation(
            engine       = engine,
            answer       = advisory_brief,
            eval_context = eval_ctx,
            diagnostics_json_path = diag_path
        )
        print_ragas_results(ragas_scores)


def main():
    print("=" * 65)
    print("  GRAP-DHAKA | POLICY-AUDITING RAG AGENT  (Week 7 — Final)")
    print("=" * 65)

    active_key = os.environ.get("GROQ_API_KEY")
    if not active_key and IN_COLAB:
        try:
            active_key = userdata.get("GROQ_API_KEY")
        except Exception:
            pass

    if not active_key:
        print("[ERROR] GROQ_API_KEY not found.")
        print("Set it in Colab Secrets (key icon in left sidebar) or run:")
        print("  import os; os.environ['GROQ_API_KEY'] = 'your_key_here'")
        return

    missing = [p for p in [NAQMP_MD_PATH, WHO_MD_PATH] if not os.path.exists(p)]
    if missing:
        print("\n[CRITICAL] Missing Markdown files:")
        for p in missing:
            print(f"  → {p}")
        print("\nUpload your .md files or ensure they exist in your workspace folder.")
        return

    # Instantiate engine (rebuild=True if starting from scratch, rebuild=False to reuse)
    engine = GRAPDhakaEngine(api_key=active_key, rebuild=False)
    print("\n[PHASE 1] Building vector database from Markdown...")
    engine.build_vector_db()
    print("\n[PHASE 1] ✅ Database build complete.")

    # Execute all three scenarios sequentially
    run_scenarios(engine)


if __name__ == "__main__":
    main()

  GRAP-DHAKA | POLICY-AUDITING RAG AGENT  (Week 7 — Final)

[PHASE 1] Building vector database from Markdown...

[DB] === Building Bangladesh Legal Corpus (Collection 2) ===

[DB-INDEX] Reading: Bangladesh National Air Quality Management Plan 2024-2030.md
[DB-INDEX] 240 semantic chunks extracted. Building local embeddings...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 62.0MiB/s]


[DB-INDEX] ✅ 'naqmp_policy' now has 239 chunks.

[DB] ✅ APCR 2022 found. Integrating Tier 2 (Codified Law)...

[DB-INDEX] Reading: Air Pollution Control Rules 2022.md
[DB-INDEX] 95 semantic chunks extracted. Building local embeddings...
[DB-INDEX] ✅ 'naqmp_policy' now has 333 chunks.
[DB]    Full legal corpus: Tier 2 (APCR) + Tier 3 (NAQMP).

[DB] === Building WHO Scientific Baseline (Collection 1) ===

[DB-INDEX] Reading: WHO global_Air_Quality_Guildlines_eng.md
[DB-INDEX] 850 semantic chunks extracted. Building local embeddings...
[DB-INDEX] ✅ 'who_guidelines' now has 842 chunks.

[PHASE 1] ✅ Database build complete.

 Running Scenario: Winter Inversion Crisis

  ENVIRONMENTAL ACTION ADVISORY BRIEF (EAAB)
## 1. Forecast Diagnostics & Precautionary Calibration  

| Item | Value | Note |
|------|-------|------|
| Modelled 24‑h PM₂.₅ (T+24 h) | **145 µg/m³** | Direct output of the XGBoost forecast. |
| 3‑Day Rolling Average PM₂.₅ | **155 µg/m³** | Indicates sustained extreme concentrati